# Demo VirtualHome

This is a demo of how to run VirtualHome UnitySimulator. The demo will walk though how to start an environment and visualize it, how to prepare it to perform activities and finally how to perform activities in them.

<img src=https://raw.githubusercontent.com/xavierpuigf/virtualhome_unity/master/doc/assets/banner.gif />



# Setup
The code below is only needed if you are running the demo in colab. It installs a package to stream content on Google Colab.

**Note:** Make sure you have GPU enabled if you are running in colab. 

Select Runtime > Change runtime type > Hardware accelerator: GPU


In [ ]:
from pathlib import Path
import os
from sys import platform

# Local repo layout. Keep these absolute so the notebook works no matter
# where Jupyter was launched from.
PROJECT_ROOT = Path("/Users/talhachafekar/Documents/social_embodied")
VH_REPO_ROOT = PROJECT_ROOT / "virtualhome"
VH_PACKAGE_ROOT = VH_REPO_ROOT / "virtualhome"
DEMO_DIR = VH_PACKAGE_ROOT / "demo"
SIM_DIR = VH_PACKAGE_ROOT / "simulation"
UNITY_SIM_DIR = SIM_DIR / "unity_simulator"

if 'google.colab' in str(get_ipython()):
    print('Running on CoLab')
    osname = "linux"
    !pip install git+https://github.com/xavierpuigf/colabstreamer
    import colabstreamer
    colabstreamer.config_all()
    _xorg = colabstreamer.open_xorg()
    !git clone https://github.com/xavierpuigf/virtualhome.git
    %cd /content/virtualhome
    !pip install -r requirements.txt
else:
    if platform == "darwin":
        osname = "macos"
    elif platform.startswith("linux"):
        osname = "linux"
    elif platform in ("windows", "win32"):
        osname = "windows"
    else:
        osname = platform
    %cd {VH_PACKAGE_ROOT}
    print("VirtualHome package root:", VH_PACKAGE_ROOT)
    print("Unity simulator dir:", UNITY_SIM_DIR)


## Download the simulator

In [ ]:
# Local setup: the macOS simulator executable is already downloaded and copied
# under virtualhome/virtualhome/simulation/unity_simulator/.
# Do not run the original Linux wget/unzip cell on macOS.
if 'google.colab' in str(get_ipython()):
    if not os.path.isfile(f"{osname}_exec.zip"):
        ! wget http://virtual-home.org/release/simulator/v2.0/v2.3.0/linux_exec.zip
        ! unzip -q linux_exec.zip

%cd {DEMO_DIR}
print("Notebook working dir:", Path.cwd())


# Imports

In [ ]:
%matplotlib notebook
import IPython.display
import glob
import os
import sys
from pathlib import Path
from sys import platform

sys.path.insert(0, str(DEMO_DIR))
sys.path.insert(0, str(SIM_DIR))

from utils_demo import *
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import virtualhome
from unity_simulator.comm_unity import UnityCommunication
from unity_simulator import utils_viz

# Keep this True while validating the notebook manually. It reduces heavy camera
# loops that can crash the macOS Unity player, but it does not skip the cat scene.
DEMO_SAFE_MODE = True
MAX_SCENES_TO_SHOW = 1 if DEMO_SAFE_MODE else 10
CAMERA_IMAGE_WIDTH = 320 if DEMO_SAFE_MODE else 640
CAMERA_IMAGE_HEIGHT = 180 if DEMO_SAFE_MODE else 360
MODALITY_IMAGE_HEIGHT = 180 if DEMO_SAFE_MODE else 320
RUN_CUSTOM_OBJECT_DEMO = True
RUN_FULL_DEMO_SECTIONS = False

CAT_ID = 1000
CAT_PREFAB_NAME = 'Cat_1'
CAT_PREFABS_MAP = {'cat': [CAT_PREFAB_NAME]}


def _reset_scene(comm, scene_id):
    result = comm.reset(scene_id)
    if isinstance(result, tuple):
        success = result[0]
        message = result[1] if len(result) > 1 else ''
    else:
        success = bool(result)
        message = ''
    if not success:
        raise RuntimeError(f"reset({scene_id}) failed: {message}")
    return success

# Prefer the v2.2.4 macOS app for this social embodied setup. v2.3.0 launches,
# but in our local tests its API camera images came back gray.
MACOS_EXEC_224 = UNITY_SIM_DIR / "macos_exec.2.2.4.app" / "Contents" / "MacOS" / "VirtualHome"
MACOS_EXEC_230 = UNITY_SIM_DIR / "macos_exec.v2.3.0.app" / "Contents" / "MacOS" / "VirtualHome"
DEFAULT_SIM_EXEC = MACOS_EXEC_224 if MACOS_EXEC_224.exists() else MACOS_EXEC_230

def _select_camera_ids(comm, ids):
    success, ncameras = comm.camera_count()
    if not success:
        raise RuntimeError(f"camera_count failed: {ncameras}")
    selected = []
    for camera_id in ids:
        resolved = camera_id if camera_id >= 0 else ncameras + camera_id
        if 0 <= resolved < ncameras:
            selected.append(resolved)
    if not selected:
        raise RuntimeError(f"No valid camera ids selected from {ids}; simulator has {ncameras} cameras")
    return selected

# These are fixed environment cameras, not the action/render cameras. Avoid only
# using -1 because that is usually the top-down overview camera.
SCENE_PREVIEW_CAMERA_IDS = [0, 8, 16, 24, 32, 40] if DEMO_SAFE_MODE else [3, 32, -5, -1, -20, 15, 48, -8, 50, 17]
CAT_PREVIEW_CAMERA_IDS = [0, 8, 16, 24, 32, 40] if DEMO_SAFE_MODE else range(-6, 0)

def get_scene_cameras(comm, ids, mode='normal', image_width=CAMERA_IMAGE_WIDTH, image_height=CAMERA_IMAGE_HEIGHT):
    cameras_select = _select_camera_ids(comm, list(ids))
    success, imgs = comm.camera_image(cameras_select, mode=mode, image_width=image_width, image_height=image_height)
    if not success:
        raise RuntimeError(f"camera_image failed for cameras {cameras_select}: {imgs}")
    return imgs

def display_scene_cameras(comm, ids, nrows=1, mode='normal', image_width=CAMERA_IMAGE_WIDTH, image_height=CAMERA_IMAGE_HEIGHT):
    imgs = get_scene_cameras(comm, ids, mode=mode, image_width=image_width, image_height=image_height)
    return display_grid_img(imgs, nrows=nrows)

def display_scene_modalities(comm, ids, modalities=['normal', 'seg_class', 'seg_inst', 'depth'], nrows=1):
    cameras_select = _select_camera_ids(comm, list(ids))
    imgs_modality = []
    for mode_name in modalities:
        success, imgs = comm.camera_image(cameras_select, mode=mode_name, image_width=CAMERA_IMAGE_WIDTH, image_height=MODALITY_IMAGE_HEIGHT)
        if not success:
            raise RuntimeError(f"camera_image failed for mode={mode_name}, cameras={cameras_select}: {imgs}")
        imgs_modality += imgs
    return display_grid_img(imgs_modality, nrows=nrows)


# Starting communication

The first step is to start a communication with the simulator. Make sure before you run this that you have downloaded the simulator, and placed it under the `simulation` folder. You will be interacting with the simulator with the communication `comm` created here. You can include the file name of the simulator or just call `UnityCommunication()` and manually open the executable.

Select `manual` if you are opening the executable separately, and `auto` if the unity executable is still not open.

Remember that if you are running this in a headless server, you will need to start a display in a separate terminal using: 

```
sudo python helper_scripts/startx $DISPLAY_NUM
```

This is not needed if you are running a colab notebook.

In [ ]:
# manual: launch the Unity app yourself, then connect to it on port 8080.
# auto: let UnityCommunication launch DEFAULT_SIM_EXEC.
mode = 'manual'  # auto / manual
PORT = "8080"
TIMEOUT_WAIT = 120

if mode == 'auto':
    if not DEFAULT_SIM_EXEC.exists():
        raise FileNotFoundError(f"Simulator executable not found: {DEFAULT_SIM_EXEC}")
    comm = UnityCommunication(file_name=str(DEFAULT_SIM_EXEC), port=PORT, timeout_wait=TIMEOUT_WAIT)
else:
    if platform == 'darwin':
        print("If Unity is not already running, launch it in a terminal with:")
        print(f"/usr/bin/open -n {DEFAULT_SIM_EXEC.parents[2]} --args -screen-fullscreen 0 -screen-quality 4")
    else:
        print("If Unity is not already running, launch it in a terminal with:")
        print(f"{DEFAULT_SIM_EXEC} -screen-fullscreen 0 -screen-quality 4")
    comm = UnityCommunication(port=PORT, timeout_wait=TIMEOUT_WAIT)

try:
    success, ncameras = comm.camera_count()
    print("Connected UnityCommunication on port", PORT, "with", ncameras, "cameras")
except Exception as exc:
    raise RuntimeError(
        f"Unity simulator is not responding on 127.0.0.1:{PORT}. "
        "Launch the simulator with the command printed above, wait until the apartment window appears, "
        "then rerun this cell."
    ) from exc


# Starting and Visualizing Scenes

After initalizing the simulation. We can interact with the environments provided in VirtualHome. The simulator is composed of 50 human designed apartments, a sample of environments can be seen here.

In [ ]:
# The environments are numbered 0 to 50. In safe mode, show one scene from
# several fixed cameras. Camera -1 is the top-down overview, so we avoid using
# only that camera.
views = []
for scene_id in tqdm(range(MAX_SCENES_TO_SHOW)):
    comm.reset(scene_id)
    comm.remove_terrain()
    scene_views = get_scene_cameras(comm, SCENE_PREVIEW_CAMERA_IDS)
    views += scene_views

IPython.display.display(display_grid_img(views, nrows=2 if DEMO_SAFE_MODE else 2))


# Procedural Generation

VirtualHome also has support for procedural generation where we can generate completely new environments during runtime.

In [ ]:
views = []
for proc_gen_seed in tqdm(range(MAX_SCENES_TO_SHOW)):
    comm.procedural_generation(proc_gen_seed)
    comm.remove_terrain()
    scene_views = get_scene_cameras(comm, SCENE_PREVIEW_CAMERA_IDS)
    views += scene_views

IPython.display.display(display_grid_img(views, nrows=2 if DEMO_SAFE_MODE else 2))


## Scene start and display

We will start scene number 4 and visualize it from different views. We start it by calling reset. Scenes are numbered from 0 to 49.

In [ ]:
comm.reset(0)

Each scene has multiple cameras, we will take screenshots for some of the cameras in this scene, specified by indices.

In [ ]:
indices = SCENE_PREVIEW_CAMERA_IDS
img_final = display_scene_cameras(comm, indices, nrows=2 if DEMO_SAFE_MODE else 2)
IPython.display.display(img_final)


## VirtualHome supports multiple modalities

The cameras can also display other modalities, such as semantic segmentation, depth, instance segmentation or optical flow when playing videos. We will display a few of those here.

In [ ]:
indices = [0] if DEMO_SAFE_MODE else [-20, 1, 48, -8, 17]
modalities = ['normal'] if DEMO_SAFE_MODE else ['normal', 'seg_class', 'seg_inst']
img_final = display_scene_modalities(comm, indices, modalities=modalities, nrows=1 if DEMO_SAFE_MODE else 5)
IPython.display.display(img_final)


## Including cameras

You can also add new cameras in the scene and get screenshots from those

In [ ]:
# Add a custom static scene camera and keep its id. The new camera id is the
# camera count before adding, because Unity appends it to the camera list.
success, camera_count_before = comm.camera_count()
if not success:
    raise RuntimeError(f"camera_count failed before add_camera: {camera_count_before}")
custom_camera_id = camera_count_before

success, message = comm.add_camera(position=[-3, 1.8, -4], rotation=[20, 120, 0], field_view=60)
print("add_camera:", success, message)
if not success:
    raise RuntimeError(f"add_camera failed: {message}")

img_final = display_scene_cameras(comm, [custom_camera_id], nrows=1)
IPython.display.display(img_final)
print("custom_camera_id:", custom_camera_id)


We can also update existing cameras, here we will update the camera we just added

In [ ]:
# v2.2.4 macOS does not implement the update_camera action even though the
# Python client exposes it. To demonstrate a changed viewpoint, add a second
# custom camera with the new pose and render that camera directly.
success, camera_count_before = comm.camera_count()
if not success:
    raise RuntimeError(f"camera_count failed before adding replacement camera: {camera_count_before}")
updated_custom_camera_id = camera_count_before

success, message = comm.add_camera(
    position=[-1.5, 1.6, -5.5],
    rotation=[12, 35, 0],
    field_view=80,
)
print("add replacement camera:", success, message)
if not success:
    raise RuntimeError(f"replacement add_camera failed: {message}")

img_final = display_scene_cameras(comm, [updated_custom_camera_id], nrows=1)
IPython.display.display(img_final)
print("updated_custom_camera_id:", updated_custom_camera_id)


## Visualizing the scene as a graph

Each scene in VirtualHome can be visualized as a graph, allowing to query the objects appearing, and their relationships. We start by obtaining the graph from the current scene.

In [ ]:
s, graph = comm.environment_graph()

The graph is a dictionary with `nodes` and `edges`. Each node corresponds to an object and contains information such as.
- class_name: the object_name
- states: in which state the object is
- id: a number you can use to perform actions over the object 

Let's print one of the nodes, to see more of the information

In [ ]:
graph['nodes'][140]

The edges connect object ids with spatial relationships, such as `INSIDE`, `ON`, `CLOSE`. You can check more of them in the `simulation` folder.

In [ ]:
graph['edges'][:5]

The graph also contains bounding box and center information, which may be useful to reason about the environment layout.

# Modifying your environment and preparing for activities

In the previous section we viewed how to read and visualize the environment. Now we are interested in modifying the environment to perform activities in them. 

## Get default environment

All the environments have a default setting. We can go to this setting by calling reset()

In [ ]:
comm.reset(4)

## Adding Objects

We will start by adding objects to interact with in the environments. We can start by adding a cat in the environment.

### Adding a cat

We first want to make sure that the cat will be added in the current environment. Let's say that we want to add it in one of the sofas.

In [ ]:
# Capture several views before adding the cat. A single camera can easily miss a
# small object, so use the cat preview camera set for before/after comparison.
imgs_prev = get_scene_cameras(comm, CAT_PREVIEW_CAMERA_IDS)
IPython.display.display(display_grid_img(imgs_prev, nrows=2 if DEMO_SAFE_MODE else 2))


We start by reading the graph and looking for one of the sodas in the scene.

In [ ]:
comm.reset(4)
success, graph = comm.environment_graph()
if not success:
    raise RuntimeError("environment_graph failed")

# Build the cat scene from a clean environment graph. This avoids stale character
# and HOLDS_* edges from earlier cells poisoning expand_scene.
graph = clean_graph(graph)
sofas = find_nodes(graph, class_name='sofa') or find_nodes(graph, class_name='couch') or find_nodes(graph, class_name='love_seat')
if not sofas:
    raise RuntimeError("Could not find a sofa/couch/love_seat node in scene 4")
sofa = sofas[-1]
print("cat placement surface:", sofa)


We now add one node with id `1000` of type cat, and an edge between the sofa node and the cat, specifying that the cat should be on the sofa.

In [ ]:
graph['nodes'] = [node for node in graph['nodes'] if node['id'] != CAT_ID]
graph['edges'] = [edge for edge in graph['edges'] if edge.get('from_id') != CAT_ID and edge.get('to_id') != CAT_ID]

add_node(graph, {
    'class_name': 'cat',
    'category': 'Animals',
    'id': CAT_ID,
    'prefab_name': CAT_PREFAB_NAME,
    'properties': ['GRABBABLE'],
    'states': []
})
add_edge(graph, CAT_ID, 'ON', sofa['id'])
cat_id = CAT_ID
cat_graph = graph
print(f"Added cat id={cat_id} prefab={CAT_PREFAB_NAME} ON {sofa['class_name']} id={sofa['id']}")


#### Update environment

The graph is now updated, but now we have to call the simulator so that the environment gets updated with the graph. Let's do it.

In [ ]:
cat_prefabs_map = {'cat': [CAT_PREFAB_NAME]}
print("using cat_prefabs_map:", cat_prefabs_map)
success, message = comm.expand_scene(cat_graph, prefabs_map=cat_prefabs_map)
print(success, message)
if not success:
    raise RuntimeError(f"expand_scene failed for cat graph: {message}")


You can now take an image of the environment, you will see how there has been a cat added in the environment

In [ ]:
success, graph_after_cat = comm.environment_graph()
if not success:
    raise RuntimeError("environment_graph failed after cat expand_scene")

cats_after = [node for node in graph_after_cat['nodes'] if node['class_name'] == 'cat']
cat_edges_after = [
    edge for edge in graph_after_cat['edges']
    if edge.get('from_id') in [node['id'] for node in cats_after]
    or edge.get('to_id') in [node['id'] for node in cats_after]
]
print("cats in Unity graph:", cats_after)
print("cat edges in Unity graph:", cat_edges_after)
if not cats_after:
    raise RuntimeError("expand_scene succeeded, but no cat node was found in Unity's environment graph")

# Unity may remap the requested id=1000 to a new scene id. Use the real returned
# id from here on, especially for scripts.
cat = cats_after[0]
cat_id = cat['id']
target_class, target_id = cat['class_name'], cat_id
cat_center = cat.get('bounding_box', {}).get('center') or cat.get('obj_transform', {}).get('position')
print("using cat:", target_class, target_id, "center:", cat_center)

imgs_final = get_scene_cameras(comm, CAT_PREVIEW_CAMERA_IDS)
print("Before adding cat:")
IPython.display.display(display_grid_img(imgs_prev, nrows=2 if DEMO_SAFE_MODE else 2))
print("After adding cat, wide scene cameras:")
IPython.display.display(display_grid_img(imgs_final, nrows=2 if DEMO_SAFE_MODE else 2))

# Add close-up cameras around the actual cat position. The standard room cameras
# can easily miss a small object on a sofa.
cat_close_camera_ids = []
if cat_center:
    x, y, z = cat_center
    camera_height = max(1.0, y + 0.65)
    close_camera_specs = [
        ([x, camera_height, z - 2.4], [12, 0, 0], 45),
        ([x - 1.3, camera_height, z - 1.4], [10, 45, 0], 45),
        ([x + 1.3, camera_height, z - 1.4], [10, 315, 0], 45),
    ]
    for position, rotation, field_view in close_camera_specs:
        success, camera_count_before = comm.camera_count()
        if not success:
            raise RuntimeError(f"camera_count failed before cat close camera: {camera_count_before}")
        new_camera_id = camera_count_before
        success, message = comm.add_camera(position=position, rotation=rotation, field_view=field_view)
        print("add cat close camera:", success, new_camera_id, position, rotation, message)
        if success:
            cat_close_camera_ids.append(new_camera_id)

if cat_close_camera_ids:
    cat_closeups = get_scene_cameras(comm, cat_close_camera_ids)
    print("After adding cat, close-up cameras:")
    IPython.display.display(display_grid_img(cat_closeups, nrows=1))

# Segmentation is often easier than RGB for confirming small inserted objects.
seg_ids = cat_close_camera_ids[:3] if cat_close_camera_ids else CAT_PREVIEW_CAMERA_IDS[:3]
seg_final = get_scene_cameras(comm, seg_ids, mode='seg_class')
print("After adding cat, segmentation views:")
IPython.display.display(display_grid_img(seg_final, nrows=1))


In [ ]:
imgs_prev = imgs_final

### Opening fridge

We may not want to add any new object, but just to change the state of the current objects. We can do this very similarly, by changing the environment graph. Let's say we want to open the fridge.

We read again the graph

In [ ]:
success, graph = comm.environment_graph()

We find the node `fridge` and change its `states` to open. 

In [ ]:
fridge = find_nodes(graph, class_name='fridge')[0]

In [ ]:
fridge['states'] = ['OPEN']

We finally expand the graph, as we did before.

In [ ]:
success, message = comm.expand_scene(graph)
print(success, message)


In [ ]:
imgs_final = get_scene_cameras(comm, [-4])
display_grid_img(imgs_prev+imgs_final, nrows=1)

### Appliances

We will use the same method as before to change the state of some appliances. Again just by modifying the state of the graph.

We take a picture of the apartment to see how it looks before doing any change

In [ ]:
indices = [0]
imgs_prev = get_scene_cameras(comm, indices)
display_grid_img(imgs_prev, nrows=1)

We now get the graph of the environment, and select a TV and a light

In [ ]:
success, graph = comm.environment_graph()
prev_graph = graph
tv_node = [x for x in graph['nodes'] if x['class_name'] == 'tv'][0]
light_node = [x for x in graph['nodes'] if x['class_name'] == 'lightswitch'][0]

We change the state and modify the scene with the new graph.

In [ ]:
tv_node['states'] = ['ON']
light_node['states'] = ['OFF']
success, message = comm.expand_scene(graph)
print(success, message)
last_graph = graph


We visualize the final scene

In [ ]:
imgs_final = get_scene_cameras(comm, indices)
display_grid_img(imgs_prev+imgs_final, nrows=1)

### Setting up time

VirtualHome also includes a real-time management system based on the 24 hour system and assign tasks to agents depending on the time of day. Furthermore the time also affects the sun's position in the sky which has a direct effect on the outdoor lighting and indoor lighting through the windows.

In [ ]:
comm.reset(2)

In [ ]:
views = []
s, message = comm.add_camera(position=[-9.2,1.3,-3], rotation=[15, 130, 0], field_view=60)
cam_id = int(message.split(':')[1])
# Set time to 05:30 
comm.set_time(hours=10, minutes=30, seconds=0)
morning_view = get_scene_cameras(comm, [cam_id])
views += morning_view

# Set time to 8:30 
comm.set_time(hours=15, minutes=30, seconds=0)
day_view = get_scene_cameras(comm, [cam_id])
views += day_view

# Set time to 21:00 
comm.set_time(hours=21, minutes=0, seconds=0)
night_view = get_scene_cameras(comm, [cam_id])
views += night_view
    
IPython.display.display(display_grid_img(views, nrows=1)) 

We can also deactivate the time and the forests outside, if we want

In [ ]:
comm.reset(0)
comm.remove_terrain()
no_day_view = get_scene_cameras(comm, [17])
IPython.display.display(display_grid_img(no_day_view, nrows=1)) 

# Generating Scripts

We now can start scenes, visualize them and modify them. The last step is to perform activities in them. We do this by defining scripts: Lists of instructions that will be executed in sequence. Each instruction contains an action, an object, and an id. The id should match with the `id` of each of the nodes in the environment graph.

You can check the list of actions currently implemented [here](https://github.com/xavierpuigf/virtualhome/tree/master/simulation#actions)

### Adding a character

The first step is to add agents in the environment, that will be performing the activity. You can specify which agent you want to add and the room where you want to add it

In [ ]:
# Prepare the current cat scene for rendering. If the cat is already in Unity,
# do not expand the graph again; repeated expand_scene calls can duplicate cats.
success, g = comm.environment_graph()
if not success:
    raise RuntimeError("environment_graph failed before cat script setup")

cats = [node for node in g['nodes'] if node['class_name'] == 'cat']
if not cats:
    if 'cat_graph' not in globals():
        raise RuntimeError("Run the cat insertion cells first so cat_graph is defined.")
    cat_prefabs_map = {'cat': [CAT_PREFAB_NAME]}
    success, message = comm.expand_scene(cat_graph, prefabs_map=cat_prefabs_map)
    print("cat expand_scene:", success, message)
    if not success:
        raise RuntimeError(f"expand_scene failed for cat graph: {message}")
    success, g = comm.environment_graph()
    if not success:
        raise RuntimeError("environment_graph failed after cat expand_scene")
    cats = [node for node in g['nodes'] if node['class_name'] == 'cat']

if not cats:
    raise RuntimeError("Cat is missing; cannot run the cat scenario.")

characters = [node for node in g['nodes'] if node['class_name'] == 'character']
if not characters:
    success, message = comm.add_character('Chars/Female2', initial_room='kitchen')
    print("add_character:", success, message)
    if not success:
        raise RuntimeError(f"add_character failed: {message}")
    success, g = comm.environment_graph()
    if not success:
        raise RuntimeError("environment_graph failed after add_character")
else:
    print("character already present:", [(node['id'], node.get('prefab_name')) for node in characters])

sofas = find_nodes(g, class_name='sofa') or find_nodes(g, class_name='couch') or find_nodes(g, class_name='love_seat')
if not sofas:
    raise RuntimeError("Could not find sofa/couch/love_seat after adding character")
sofa = sofas[-1]

cats = [node for node in g['nodes'] if node['class_name'] == 'cat']
cat = cats[0]
cat_id = cat['id']
target_class, target_id = cat['class_name'], cat_id

print("cat candidates:", [(node['id'], node.get('bounding_box', {}).get('center')) for node in cats])
print("sofa:", sofa['class_name'], sofa['id'])
print("target:", target_class, target_id)


If you count the number of cameras, you will see that a few new cameras have been added into the scene. These are cameras attached to the character. When the character moves, the cameras will move as well.

In [ ]:
imgs_prev = get_scene_cameras(comm, CAT_PREVIEW_CAMERA_IDS)
display_grid_img(imgs_prev, nrows=2 if DEMO_SAFE_MODE else 2)


## Generating the first script

Let's start by interacting witht the cat and the sofa that we set up before. The cat had id 1000. The sofa was stored in a variable `sofa` containing that node. We can query its id directly. This sequence will make the agent walk to the sofa, grab the cat and sit in the sofa.

In [ ]:
script = [
    '<char0> [Walk] <{}> ({})'.format(target_class, target_id),
    '<char0> [Find] <{}> ({})'.format(target_class, target_id),
    '<char0> [Grab] <{}> ({})'.format(target_class, target_id),
    '<char0> [Walk] <{}> ({})'.format(sofa['class_name'], sofa['id']),
    '<char0> [Sit] <{}> ({})'.format(sofa['class_name'], sofa['id']),
]
script


We now want to execute the script in the environment. We do that through render_script. Notice that we can specify a file name, which will be used to save a video with the activity.

In [ ]:
success, message = comm.render_script(script=script,
                                      processing_time_limit=60,
                                      find_solution=True,
                                      image_width=CAMERA_IMAGE_WIDTH,
                                      image_height=CAMERA_IMAGE_HEIGHT,
                                      frame_rate=5,
                                      skip_animation=False,
                                      recording=True,
                                      save_pose_data=True,
                                      image_synthesis=['normal'],
                                      camera_mode=['FIRST_PERSON', 'PERSON_TOP'],
                                      file_name_prefix='relax')
print(success, message)


This saves the frames of the video into the `Output/relax` folder, which should be where you had your executable. Let's generate a video from the frames.

In [ ]:
# Rendered frames are saved under the simulator Output directory.
path_video = str(UNITY_SIM_DIR / "Output")
utils_viz.generate_video(input_path=path_video, prefix='relax', output_path='.')


In [ ]:
video_path = Path('./video_normal.mp4')
if video_path.exists():
    display_vid(str(video_path))
else:
    print("Video was not found at", video_path.resolve())
    print("Check rendered frames under", UNITY_SIM_DIR / "Output" / "relax")


Other paramters to render_script are:
- script: a list of script lines
- randomize_execution: randomly choose elements
- random_seed: random seed to use when randomizing execution, -1 means that the seed is not set
- find_solution: find solution (True) or use graph ids to determine object instances (False)
- processing_time_limit: time limit for finding a solution
- skip_execution: skip rendering, only check if a solution exists
- output_folder: folder to output renderings, default is Output/
- file_name_prefix: prefix of created files (screenshots are put to output_folder/file_name_prefix/)
- frame_rate: frame rate
- capture_screenshot: save screenshots
- image_synthesis: save depth, segmentation, flow images
- save_pose_data: save pose data
- save_scene_states: save scene states
- character_resource: path to character resource to be used
- camera_mode: automatic (AUTO), first person (FIRST_PERSON), top (PERSON_TOP), front person view (PERSON_FRONT)

In [ ]:
# We can also visualize the skeleton if pose data was generated.
path_video = str(UNITY_SIM_DIR / "Output")
skeleton_file = Path(path_video) / 'relax' / '0' / 'pd_relax.txt'
if skeleton_file.exists():
    pose_char, frames = utils_viz.get_skeleton(input_path=path_video, prefix='relax')
    ax = plt.axes(projection='3d')
    frame = min(40, pose_char.shape[0] - 1)
    center_char = pose_char.mean(1)
    ax.scatter3D(pose_char[frame, :, 0], pose_char[frame, :, 2], pose_char[frame, :, 1])
    ax.set_xlim(center_char[frame, 0]-0.8, center_char[frame, 0] + 0.8)
    ax.set_ylim(center_char[frame, 2]-0.8, center_char[frame, 2] + 0.8)
else:
    print("Skeleton pose file was not found at", skeleton_file)


## Generating a script without a video

In some occasions you may not be interested in generating a full video for the script, for example if you want to do RL. You can run the script withtout rendering a video by setting image_synthesis to empty. This will execute the script much more quickly

Restart the previous graph

In [ ]:
comm.reset(4)
if 'prev_graph' in globals():
    try:
        comm.expand_scene(prev_graph)
    except Exception as exc:
        print("Skipping prev_graph restore after expand_scene error:", exc)
comm.add_character()
s, g = comm.environment_graph()
print(s, len(g.get('nodes', [])) if s else None)


Run with `image_synthesis=[]`

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    success, message = comm.render_script(script=script,
                                          processing_time_limit=60,
                                          find_solution=False,
                                          image_synthesis=[])
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


## Generating from multiple views

We can chose which camera to use while rendering the videos. This is done through the flag `CAMERA_MODE`.

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    comm.reset(4)
    if 'last_graph' in globals():
        try:
            comm.expand_scene(last_graph)
        except Exception as exc:
            print("Skipping last_graph restore after expand_scene error:", exc)
    comm.add_character()
    success, message = comm.render_script(script=script,
                                          processing_time_limit=60,
                                          find_solution=True,
                                          recording=True,
                                          image_width=CAMERA_IMAGE_WIDTH,
                                          image_height=CAMERA_IMAGE_HEIGHT,
                                          image_synthesis=['normal'],
                                          file_name_prefix='multiview',
                                          camera_mode=['FIRST_PERSON', 'PERSON_TOP'])
    print(success, message)
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


You can specify camera names, for cameras that will follow the character or camera indices, for static cameras. To get the list of camera names, call character_cameras:

In [ ]:
comm.character_cameras()[1]

## Generating underspecified videos

If we do not care which objects the simulator should interact with, we can also let it decide. If we use the flag `find_solution=True` we can start enumerating objects by `1` instead of following the graph ids. Unity will try to find a solution. Note that if we want to interact with 2 objects of the same kind, they will need to have different ids (i.e. 1 and 2)

In [ ]:
comm.reset(0)

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    s, g = comm.environment_graph()
    [node for node in g['nodes'] if node['class_name'] == 'salmon']
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


In [ ]:
comm.add_character()

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    script = ['<char0> [walk] <salmon> (1)',
              '<char0> [grab] <salmon> (1)',
              '<char0> [walk] <microwave> (1)',
              '<char0> [open] <microwave> (1)',
              '<char0> [putin] <salmon> (1) <microwave> (1)',
              '<char0> [close] <microwave> (1)',
              '<char0> [switchon] <microwave> (1)']
          
    success, message = comm.render_script(script=script, 
                                          find_solution=True,
                                          processing_time_limit=80,
                                          frame_rate=15,
                                          image_width=512, image_height=320,
                                          skip_animation=False,
                                          image_synthesis=['normal'],
                                          camera_mode=['PERSON_FROM_BACK'],
                                          recording=True,
                                          file_name_prefix='milk')
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


# Multi-agent Actions

You can also generate actions with multiple agents in them, simply add more agents, and specify which agents should do which action

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    # Reset the scene
    comm.reset(0)
    s, g = comm.environment_graph()
    # Add two agents this time
    comm.add_character('Chars/Male2', initial_room='kitchen')
    comm.add_character('Chars/Female4', initial_room='bedroom')

    # Get nodes for salmon and microwave, glass, faucet and sink
    salmon_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'salmon'][0]
    microwave_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'microwave'][0]
    glass_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'waterglass'][-1]
    sink_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'sink'][0]
    faucet_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'faucet'][-1]


    # Put salmon in microwave
    script = [
        '<char0> [walk] <salmon> ({}) | <char1> [walk] <glass> ({})'.format(salmon_id, glass_id),
        '<char0> [grab] <salmon> ({}) | <char1> [grab] <glass> ({})'.format(salmon_id, glass_id),
        '<char0> [open] <microwave> ({}) | <char1> [walk] <sink> ({})'.format(microwave_id, sink_id),
        '<char0> [putin] <salmon> ({}) <microwave> ({}) | <char1> [putback] <glass> ({}) <sink> ({})'.format(salmon_id, microwave_id, glass_id, sink_id),
        '<char0> [close] <microwave> ({}) | <char1> [switchon] <faucet> ({})'.format(microwave_id, faucet_id)
    ]
    comm.render_script(script, frame_rate=10, camera_mode=["PERSON_FROM_BACK"], recording=True)
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


# Interactive agents


So far we have seen how to generate videos, but we can use the same command to deploy or train agents in the environment. You can execute the previous instructions one by one, and get an observation or graph at every step. or that, you don't need to generate videos or have animations, since it will slow down your agents. Use `skip_animation=True` to generate actions without animating them. Remember to turn off the recording mode as well.

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    # Reset the scene
    comm.reset(0)

    # Add two agents this time
    comm.add_character('Chars/Male2', initial_room='kitchen')
    comm.add_character('Chars/Female4', initial_room='bedroom')

    # Get nodes for salmon and microwave, glass, faucet and sink
    salmon_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'salmon'][0]
    microwave_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'microwave'][0]
    glass_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'waterglass'][-1]
    sink_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'sink'][0]
    faucet_id = [node['id'] for node in g['nodes'] if node['class_name'] == 'faucet'][-1]


    # Put salmon in microwave
    script = [
        '<char0> [walk] <salmon> ({}) | <char1> [walk] <glass> ({})'.format(salmon_id, glass_id),
        '<char0> [grab] <salmon> ({}) | <char1> [grab] <glass> ({})'.format(salmon_id, glass_id),
        '<char0> [open] <microwave> ({}) | <char1> [walk] <sink> ({})'.format(microwave_id, sink_id),
        '<char0> [putin] <salmon> ({}) <microwave> ({}) | <char1> [putback] <glass> ({}) <sink> ({})'.format(salmon_id, microwave_id, glass_id, sink_id),
        '<char0> [close] <microwave> ({}) | <char1> [switchon] <faucet> ({})'.format(microwave_id, faucet_id)
    ]

    s, cc = comm.camera_count()
    images = []
    for script_instruction in script:
        print(script_instruction)
        comm.render_script([script_instruction], recording=False, skip_animation=True)
        # Here you can get an observation, for instance
        s, im = comm.camera_image([cc-8], image_width=300, image_height=300)
        images.append(im[0])
    
    display_grid_img(images, nrows=1)
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


# Finer control

We can also have finer control where objects go via action modifiers, the demo below shows how to place an object in different parts of the table

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    comm.reset(0)
    s, g = comm.environment_graph()
    # Remove stuff from the table and add the character
    table = [node for node in g['nodes'] if node['class_name'] == 'kitchentable'][0]
    mouse = [node for node in g['nodes'] if node['class_name'] == 'mouse'][0]
    mouse_id = mouse['id']
    table_id = table['id']

    objects_on_table = [edge['from_id'] for edge in g['edges'] 
                        if edge['to_id'] == table['id'] and edge['relation_type'] == 'ON']
    new_graph = {
        'nodes': [node for node in g['nodes'] if node['id'] not in objects_on_table],
        'edges': [edge for edge in g['edges'] if edge['from_id'] not in objects_on_table and 
                                                 edge['to_id'] not in objects_on_table]
    }

    comm.expand_scene(new_graph)
    comm.add_character()

    # This is to get a camera view on top of the character
    cc = comm.camera_count()[1] - 7


    # Start grabbing the mouse and go to the table
    script1 = [f'<char0> [grab] <mouse> ({mouse_id})', 
               f'<char0> [walk] <kitchentable> ({table_id})']
    s, m = comm.render_script(script1, skip_animation=True)

    images = []

    # Place the mouse in some position
    posx, posy = -1.1, -5.5
    script2 = [f'<char0> [putback] <mouse> ({mouse_id}) <kitchentable> ({table_id}) <position> {posx},{posy}']
    s, m = comm.render_script(script2, skip_animation=True)
    s, im1 = comm.camera_image([cc])
    images.append(im1[0])

    # Try a different positiom
    posx, posy = -1.3, -5.1
    script2 = [f'<char0> [grab] <mouse> ({mouse_id})',
               f'<char0> [putback] <mouse> ({mouse_id}) <kitchentable> ({table_id}) <position> {posx},{posy}']
    s, m = comm.render_script(script2, skip_animation=True)
    s, im1 = comm.camera_image([cc])
    images.append(im1[0])

    display_grid_img(images, nrows=1)
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


# Physics


VirtualHome is also able to simulate physics in the environments, and you can use the following api call to alter the gravitational force experienced in the environemnt. By activating physics, all objects behave as expected, just like in the real world. However you can also alter the "g-force" to make the agent feel like they are a astronaut in space.

In [ ]:
# Reset the scene and get the graph
comm.reset(0)
s, g = comm.environment_graph()

# Add a agent 
comm.add_character('Chars/female2', initial_room='kitchen')

# Get nodes for apple, desk, kitchen
apple = [node['id'] for node in g['nodes'] if node['class_name'] == 'apple'][0]
desk = [node['id'] for node in g['nodes'] if node['class_name'] == 'desk'][1]
kitchen = [node['id'] for node in g['nodes'] if node['class_name'] == 'kitchen'][0]


In [ ]:
# Activate gravity
comm.activate_physics()

In [ ]:
if RUN_FULL_DEMO_SECTIONS:
    # Put apple on desk
    script = [
              '<char0> [grab] <apple> ({})'.format(apple),
              '<char0> [put] <apple> ({}) <desk> ({})'.format(apple, desk),
              '<char0> [walk] <kitchen> ({})'.format(kitchen),
              '<char0> [grab] <apple> ({})'.format(apple),
              '<char0> [put] <apple> ({}) <desk> ({}  <position> 0,-3)'.format(apple, desk),
             ]
    success, message = comm.render_script(script=script[:2], 
                                          find_solution=True,
                                          processing_time_limit=80,
                                          frame_rate=15,
                                          image_width=512, image_height=320,
                                          skip_animation=False,
                                          image_synthesis=['normal'],
                                          camera_mode=['58'],
                                          recording=True,
                                          file_name_prefix='gravity')
else:
    print('Skipping this non-cat full-demo cell because RUN_FULL_DEMO_SECTIONS=False.')


In [ ]:
path_video = "./Output/"
utils_viz.generate_video(input_path=path_video, prefix='gravity', output_path='.')
